<a href="https://colab.research.google.com/github/fukagai-takuya/gifu-ai/blob/main/gifu-ai-2026-03-15/FLUX2_klein_4B_git_clone_from_fork_diffusers_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. 勉強会用に fork した diffusers のリポジトリを git clone し、ディレクトリを diffusers に変えてから、調査用に用意したブランチ investigate_flux.2_klein_4B_transformer にブランチを変えます。このブランチには調査用のブレイクポイントが `Flux2Transformer2DModel.forward` の先頭にセットされています。

In [ ]:
!git clone https://github.com/fukagai-takuya/diffusers.git
%cd diffusers
!git checkout investigate_flux.2_klein_4B_transformer

2. git clone した diffusers のソースコードを優先して参照するようにします

In [ ]:
import sys
sys.path.insert(0, "/content/diffusers/src")

3. git clone した diffusers のソースコードが優先して参照されることを確認します

In [ ]:
import diffusers
print(diffusers.__file__)

4. diffusers を使用して FLUX.2 [klein] 4B を実行する際に必要なパッケージをインストールします

In [ ]:
!pip install torch torchvision torchaudio
!pip install transformers accelerate safetensors
!pip install sentencepiece protobuf einops

5. pdb より高機能な Python デバッガ ipdb をインストールします。(investigate_flux.2_klein_4B にセットしたのは、ipdb を使用するブレイクポイントなため)

In [ ]:
!pip install ipdb

6. FLUX.2 [klein] 4B を使用した Pipeline を用意します

In [ ]:
import torch
from diffusers import Flux2KleinPipeline

dtype = torch.bfloat16

pipe = Flux2KleinPipeline.from_pretrained("black-forest-labs/FLUX.2-klein-4B", torch_dtype=dtype)
pipe.enable_model_cpu_offload()  # save some VRAM by offloading the model to CPU


7. FLUX.2 [klein] 4B を使用し、プロンプト文字列を参照して画像を生成します

In [ ]:
device = "cuda"
prompt = 'A cat holding a sign that says "Gifu AI Study Group"'

image = pipe(
    prompt=prompt,
    height=1024,
    width=1024,
    guidance_scale=1.0,
    num_inference_steps=4,
    generator=torch.Generator(device=device).manual_seed(0)
).images[0]

image.save("flux-klein.png")

8. FLUX.2 [klein] 4B を使用し、プロンプト文字列を参照して画像を生成します。height と width の値が一つ前の 7. のコードセルと異なります。

In [ ]:
device = "cuda"
prompt = 'A cat holding a sign that says "Gifu AI Study Group"'

image = pipe(
    prompt=prompt,
    height=256,
    width=512,
    guidance_scale=1.0,
    num_inference_steps=4,
    generator=torch.Generator(device=device).manual_seed(0)
).images[0]

image.save("flux-klein.png")

9. FLUX.2 [klein] 4B を使用し、入力画像とプロンプト文字列を参照して画像を生成します。下のコードでは、Web ページ上の指定した URL の 2 枚の画像を参照し、1 枚の画像を生成します。1つ目の画像を背景とし、2つ目の画像の鳥が写った写真を生成します。

In [ ]:
from diffusers.utils import load_image

device = "cuda"

url = 'https://www.leafwindow.com/wordpress-05/wp-content/uploads/2023/12/IMG_6492-20.jpg'
image1 = load_image(url)

url = 'https://www.leafwindow.com/wordpress-05/wp-content/uploads/2023/02/DSC00022-min-SonyAlpha-%E6%A8%AA.jpg'
image2 = load_image(url)

prompt = """
Use image 1 strictly as the environmental background and scene layout.
Place the bird from image 2 clearly in the foreground as the main subject.
Preserve the bird's shape, colors, and feather details from image 2.
Ensure realistic lighting consistent with the background environment.
Seamless compositing, natural shadows, accurate depth of field,
high detail, professional wildlife photography.
"""

image = pipe(
    prompt=prompt,
    image=[image1, image2],
    height=1024,
    width=1024,
    guidance_scale=1.0,
    num_inference_steps=4,
    generator=torch.Generator(device=device).manual_seed(0)
).images[0]

image.save("flux-klein-with-input-image.png")